# Get the TART positions and their antenna positions in ECEF

In [1]:
import numpy as np
import sys
import os
import json

# Get parent directory
parent_dir = os.path.abspath("..")

# Add it to sys.path
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
    
from pipeline.TART_positions import TART_positions
from pipeline.JSON_handler import to_json_safe, save_to_json

## Define a selection of TARTs to get the positions of

In [2]:
TARTS = {'namibia': 'na-unam',
          'rhodes': 'za-rhodes',
          'mauritius': 'mu-udm'}

## Use the TART API to get the station+antenna positions in ECEF

In [3]:
handler = TART_positions()

TART_positions_dict = {}

for TART in list(TARTS.keys()):
    TART_positions_dict[TART] = {}
    # Get the antenna offsets [m]
    antenna_positions = handler.get_antenna_positions(TARTS[TART])
    # Get the general TART metadata 
    TART_metadata = handler.get_TART_details(TARTS[TART])
    # Isolate the TART central position [lon, lat, alt]
    TART_position = TART_metadata['info']['location']
    TART_position = np.array([TART_position['lon'], TART_position['lat'], TART_position['alt']])
    # Convert TART position to ECEF
    TART_position_ECEF = handler.lat_long_alt_to_ECEF(TART_position)
    # Get antenna positions in ECEF by applying rotation transformation
    antennas_ecef = handler.antennas_offset_to_ECEF(TART_position_ECEF, TART_position, antenna_positions)
    TART_positions_dict[TART]['station_ecef'] = TART_position_ECEF
    TART_positions_dict[TART]['antennas_ecef'] = antennas_ecef

Antenna positions API request successful for na-unam.
TART position API request successful for na-unam.
Antenna positions API request successful for za-rhodes.
TART position API request successful for za-rhodes.
Antenna positions API request successful for mu-udm.
TART position API request successful for mu-udm.


In [4]:
TART_positions_dict

{'namibia': {'station_ecef': array([ 5608433.51629243,  1806804.26823875, -2437752.6251361 ]),
  'antennas_ecef': array([[ 5608433.65831225,  1806804.25442574, -2437752.31075326],
         [ 5608433.83369489,  1806804.33723722, -2437751.84899122],
         [ 5608433.90411162,  1806804.57334105, -2437751.51424733],
         [ 5608433.89532126,  1806804.86295066, -2437751.32112042],
         [ 5608433.83077708,  1806805.19326171, -2437751.22544044],
         [ 5608433.48437204,  1806804.57596818, -2437752.47152722],
         [ 5608433.3839104 ,  1806805.05312116, -2437752.34982063],
         [ 5608433.22958584,  1806805.42959541, -2437752.42532605],
         [ 5608433.06871843,  1806805.6746875 , -2437752.61250923],
         [ 5608432.90211683,  1806805.83143245, -2437752.87783907],
         [ 5608433.35587029,  1806804.47902131, -2437752.83656092],
         [ 5608433.11834044,  1806804.68862115, -2437753.22506765],
         [ 5608432.95289911,  1806804.68602371, -2437753.60505636],
    

## Save the positions to a JSON for future use

In [5]:
serialisable = to_json_safe(TART_positions_dict)

filepath = "/home/jdawson/repos/TART/datasets/GPS"
filename = "TART_coords.json"

save_to_json(serialisable, filepath, filename)

'JSON saved to disk.'